# LLM Comparison: LLaMA 3.3 70B vs LLaMA 3.1 8B

Compares two LLMs on the same 5 queries using identical retrieved context and prompts.

| | **LLaMA 3.3 70B** | **LLaMA 3.1 8B** |
|---|---|---|
| Groq model ID | `llama-3.3-70b-versatile` | `llama-3.1-8b-instant` |
| Developer | Meta | Meta |
| Parameters | 70 billion | 8 billion |
| Training data | ~15 trillion tokens | ~15 trillion tokens |
| Context window | 128k tokens | 128k tokens |
| Release | December 2024 | July 2024 |
| Role in this project | Primary/high-quality model | Small/fast comparison model |

Both are instruction-tuned variants from Meta's LLaMA 3 series. The 70B model has significantly more capacity for reasoning and synthesis, while the 8B model trades quality for speed and lower compute cost, which is a tradeoff relevant when scaling to many users.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import pickle
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
sys.path.insert(0, str(project_root))

from src.bm25 import BM25Retriever
from src.semantic_retriever import SemanticRetriever
from src.rag_pipeline import RAGPipeline

data_dir = project_root / 'data' / 'processed'

with open(data_dir / 'corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

bm25 = BM25Retriever()
bm25.load(str(data_dir / 'bm25_index.pkl'))
bm25.corpus = corpus

semantic = SemanticRetriever()
semantic.load(str(data_dir / 'semantic_index' / 'faiss_index'))
semantic.corpus = corpus

df = pd.read_parquet(data_dir / 'books_sample.parquet')

print(f'Project root: {project_root}')
print(f'Corpus: {len(corpus):,} documents')
print('Indexes loaded')

BM25 index loaded from /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl


/home/stoyq/miniforge3/envs/575-project/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Index loaded from /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index
Project root: /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki
Corpus: 91,641 documents
Indexes loaded


## 2. LLaMA 3.3 70B vs LLaMA 3.1 8B Comparison

Runs the same 5 queries through both models using identical retrieved context and prompt template.

Queries selected: one Easy, one Medium, three Complex.

In [ ]:
from groq import Groq

api_key = os.getenv('GROQ_API_KEY')

class LLaMA70BLLM:
    """LLaMA 3.3 70B via Groq (large model)."""
    MODEL = 'llama-3.3-70b-versatile'
    def __init__(self): self.client = Groq(api_key=api_key)
    def invoke(self, prompt):
        try:
            r = self.client.chat.completions.create(
                messages=[{'role': 'user', 'content': prompt}],
                model=self.MODEL, max_tokens=300, temperature=0.3)
            return r.choices[0].message.content
        except Exception as e:
            return f'Error: {e}'

class LLaMA8BLLM:
    """LLaMA 3.1 8B via Groq (small/fast model)."""
    MODEL = 'llama-3.1-8b-instant'
    def __init__(self): self.client = Groq(api_key=api_key)
    def invoke(self, prompt):
        try:
            r = self.client.chat.completions.create(
                messages=[{'role': 'user', 'content': prompt}],
                model=self.MODEL, max_tokens=300, temperature=0.3)
            return r.choices[0].message.content
        except Exception as e:
            return f'Error: {e}'

llama_70b = LLaMA70BLLM()
llama_8b = LLaMA8BLLM()

rag_70b = RAGPipeline(bm25, semantic, llama_70b, prompt_version="balanced")
rag_70b.corpus = corpus

rag_8b = RAGPipeline(bm25, semantic, llama_8b, prompt_version="balanced")
rag_8b.corpus = corpus

print('Large model ready:', LLaMA70BLLM.MODEL)
print('Small model ready:', LLaMA8BLLM.MODEL)
print('Prompt template: balanced')

Large model ready: llama-3.3-70b-versatile
Small model ready: llama-3.1-8b-instant
Prompt template: balanced


In [ ]:
comparison_queries = [
    ('mystery novel',                                                           'E'),
    ('book to help with anxiety',                                               'M'),
    ('best book to learn machine learning with no math background',             'C'),
    ('historical fiction set in world war 2 from a female perspective',         'C'),
    ('self help book for overcoming procrastination and building better habits', 'C'),
]

print('=' * 70)
print('LLM Comparison: LLaMA 3.3 70B vs LLaMA 3.1 8B')
print('Identical retrieval context, identical prompt template')
print('=' * 70)

for query, diff in comparison_queries:
    # Retrieve once; feed same context to both models
    doc_ids, documents = rag_70b.retrieve_hybrid(query, top_k=5)
    context = rag_70b.build_context(documents)

    answer_70b = rag_70b.generate(query, context)
    answer_8b  = rag_8b.generate(query, context)

    print(f'\nQuery [{diff}]: "{query}"')
    print('Retrieved books:')
    for rank, doc_id in enumerate(doc_ids, 1):
        title = df.iloc[doc_id]['product_title']
        print(f'  {rank}. {title}')
    print()
    print('--- LLaMA 3.3 70B (llama-3.3-70b-versatile) ---')
    print(answer_70b)
    print()
    print('--- LLaMA 3.1 8B (llama-3.1-8b-instant) ---')
    print(answer_8b)
    print('=' * 70)

LLM Comparison: LLaMA 3.3 70B vs LLaMA 3.1 8B
Identical retrieval context, identical prompt template

Query [E]: "mystery novel"
Retrieved books:
  1. Mystery at Seagrave Hall: A totally addictive cozy mystery novel (An Eve Mallow Mystery Book 3)
  2. The Harry Starke Series: Books 1-3
  3. Mystery at Apple Tree Cottage: A completely unputdownable cozy mystery novel (An Eve Mallow Mystery Book 2)
  4. Six Years
  5. Love Me If You Must (Patricia Amble Mystery Series #1)

--- LLaMA 3.3 70B (llama-3.3-70b-versatile) ---
Based on the reviews, I recommend the following mystery novels:

1. **Mystery at Seagrave Hall** (An Eve Mallow Mystery Book 3) - This book fits the request as it's described as a "totally addictive cozy mystery novel" with a surprising plot that kept the reviewer guessing.
2. **The Harry Starke Series: Books 1-3** - This series is a great match as it's mentioned to be a mystery that keeps the reader guessing, switching between suspects, which is exactly what the reviewer